# Tutorial to run the generator on a dataset of simulations

Once a set of simulated populations is created by running one of the simulator scripts (see the simulator tutorial), it is possible to generate a dataset of synthetic representations of the simulations that is readable by a machine-learning pipeline.

Suppose that you have created a set of simulated populations stored in `data/example_simulation_helper_magrot` using the `simulate_population_magrot_det.py` script.
If you want to create a dataset of 2D arrays storing the spatial density and the velocity information of the simulated neutron stars with a resolution of $32 \times 32$ and the density in the $P-\dot{P}$ diagram with a resolution of $32 \times 32$ you can run the following command:
```commandline
python pypopsyn/generator/generate_dataset_survey.py --data data/example_simulation_helper_magrot --save_dir data/example_generator_magrot --data_type array --resolution_dyn 32 --resolution_ppdot 32
```

In [ ]:
import argparse
import collections
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pathlib
import sys

import utilities.plot_settings
from pypopsyn.simulator.config_simulator import cfg
from pypopsyn.generator.generate_dataset_surveys import generate_dataset
from pypopsyn.generator.dataset_splitter import main

## Setup and run the generator

In [ ]:
output_dir = "output/generator"

In [ ]:
generator_args = argparse.Namespace(
    data = "../../data/example_simulation_helper_magrot",
    save_dir = output_dir,
    data_type = "array",
    resolution_dyn = 32,
    resolution_ppdot = 32
)
generate_dataset(generator_args)

## Visualize the generated maps

In [ ]:
ppdot_PMPS = np.load(pathlib.Path().joinpath(output_dir, "survey_PMPS_ppdot_map_0.npy"))
ppdot_HTRU = np.load(pathlib.Path().joinpath(output_dir, "survey_HTRU_ppdot_map_0.npy"))
ppdot_SMPS = np.load(pathlib.Path().joinpath(output_dir, "survey_SMPS_ppdot_map_0.npy"))
position_PMPS = np.load(pathlib.Path().joinpath(output_dir, "survey_PMPS_position_map_radec_0.npy"))
position_HTRU = np.load(pathlib.Path().joinpath(output_dir, "survey_HTRU_position_map_radec_0.npy"))
position_SMPS = np.load(pathlib.Path().joinpath(output_dir, "survey_SMPS_position_map_radec_0.npy"))

# Transpose the maps for correct visualization with imshow.
ppdot_PMPS = ppdot_PMPS.T
ppdot_HTRU = ppdot_HTRU.T
ppdot_SMPS = ppdot_SMPS.T
position_PMPS = position_PMPS.T
position_SMPS = position_SMPS.T
position_HTRU = position_HTRU.T

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

map = ax.imshow(ppdot_PMPS, cmap='viridis', origin='lower')
ax.set_xlabel("$P$ bin")
ax.set_ylabel("$\dot{P}$ bin")
colorbar = fig.colorbar(map, ax=ax)
colorbar.set_label('Number of NSs')

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

map = ax.imshow(ppdot_SMPS, cmap='viridis', origin='lower')
ax.set_xlabel("$P$ bin")
ax.set_ylabel("$\dot{P}$ bin")
colorbar = fig.colorbar(map, ax=ax)
colorbar.set_label('Number of NSs')

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

map = ax.imshow(ppdot_HTRU, cmap='viridis', origin='lower')
ax.set_xlabel("$P$ bin")
ax.set_ylabel("$\dot{P}$ bin")
colorbar = fig.colorbar(map, ax=ax)
colorbar.set_label('Number of NSs')

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

map = ax.imshow(position_PMPS, cmap='viridis', origin='lower')
ax.set_xlabel("RA bin")
ax.set_ylabel("DEC bin")
colorbar = fig.colorbar(map, ax=ax)
colorbar.set_label('Number of NSs')

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

map = ax.imshow(position_SMPS, cmap='viridis', origin='lower')
ax.set_xlabel("RA bin")
ax.set_ylabel("DEC bin")
colorbar = fig.colorbar(map, ax=ax)
colorbar.set_label('Number of NSs')

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

map = ax.imshow(position_HTRU, cmap='viridis', origin='lower')
ax.set_xlabel("RA bin")
ax.set_ylabel("DEC bin")
colorbar = fig.colorbar(map, ax=ax)
colorbar.set_label('Number of NSs')

## Split the dataset for training, validation and testing

To do this you can run the `dataset_splitter.py` script in the `pypopsyn/generator` folder.
To generate a dataset split into two subsets, one specifically for training and the other for testing, you can specify a fraction of the total dataset that will form the test subset by passing the argument `valid_split` in the `dataset_splitter` script. For example:
```commandline
python pypopsyn/generator/dataset_splitter.py --dataset_path generated_dataset --valid_split 0.2
```

This will create two files `dataset_train.csv` and `dataset_valid.csv` that will specify the samples belonging to the train dataset (80 % of the total dataset in this case) and the ones belonging to the validation dataset  (20 % of the total dataset).
We can rename the `dataset_valid.csv` into `dataset_test.csv`.
The split is performed by randomly sampling the validation subset from the total dataset according to the specified split fraction.
In this case, the `statistics_train.json` file will contain the statistics computed on the labels of the training set only.

## Setup and run the splitter

In [ ]:
splitter_args = argparse.Namespace(
    dataset_path = output_dir,
    valid_split = 0.2,
    test_split = None
)
main(splitter_args)